## Monte Carlo simulations


In [34]:
# imports 
import warnings
import numpy as np
import pandas as pd
import sklearn
from joblib import Parallel, delayed
from scipy.special import expit
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings("ignore")
theta0= 3.0 # match chang
eps = 1e-6 # error tolerance

# import modules with first steps estimators and estimators and diagnostics
import first_steps as fs
import estimators as est
import diagnostics as diag
import runs as runs

# learners
learners = {"plugin": (fs.logit_plugin, fs.lasso_plugin), 
            "cv": (fs.logit_cv, fs.lasso_cv), 
            "ridge": (fs.logit_ridge, fs.ridge),
            "mle": (fs.logit_mle, fs.ols), 
            "const": (fs.const, fs.const)}

# designs
import designs.d1_chang_dgp as d1
import designs.d2_lose_sparsitt as d2
import designs.d3_chang_aligned as d3
import designs.d4_sparse_dense_nuisance as d4
import designs.d5_n_d6_dummies_n_cats as d5_d6

## Design 1: recreating Chang's estimator

In [ ]:
if __name__ == "__main__":
    pd.set_option("display.width", 200)
    kw = dict(R=50, n_jobs=1)
    print(runs.summarize(d1.design_d1(Ks=(5,), 
                                      N_trend=(200,), **kw), 
                                      by=("design", "gamma", "K", "N", "est")).round(3))

                                    bias     sd  cover   rmse  se_sd  bias_sd     B1  err_g  err_l  sel_g  sel_l  kappa
design gamma K N   est                                                                                                 
chang  code  5 200 dml_plugin     -0.004  0.020   0.96  0.020  1.129   -0.184  0.000  0.220  0.009  0.984  0.000  0.047
                   or_did          0.019    NaN   1.00  0.019    NaN      NaN    NaN    NaN    NaN    NaN    NaN    NaN
                   oracle         -0.003  0.024   1.00  0.024  0.978   -0.140    NaN    NaN    NaN    NaN    NaN    NaN
                   oracle_support -0.005  0.028   0.96  0.028  1.091   -0.168    NaN    NaN    NaN    NaN    NaN    NaN
                   unadjusted     -0.001  0.016   1.00  0.015  1.126   -0.072    NaN    NaN    NaN    NaN    NaN    NaN
       paper 5 200 dml_plugin     -0.002  0.016   1.00  0.016  1.376   -0.146  0.000  0.159  0.008  0.912  0.000  0.056
                   or_did          0.030

## Design 2: 
This designs asks how the bias behaves as the nuisances lose sparsity (regime 2, Props 6–8). The default is polynomial decay, $γ_j ∝ j^{−a}$ with unit norm and $δ = γ$

In [ ]:
if __name__ == "__main__":
    pd.set_option("display.width", 200)
    kw = dict(R=50, n_jobs=1)
    print(runs.summarize(d2.design_d2(N_grid=(250, 500), pars=(2.0, 0.5), **kw), by=("est", "par", "cos", "N")).round(3))

                         bias     sd  cover   rmse  se_sd  bias_sd  naive     B1  err_g  err_l  sel_g  sel_l  kappa
est        par cos N                                                                                               
dml_plugin 0.5 1.0 250  0.784  0.188   0.06  0.806  0.982    4.162  0.826  0.820  0.212  1.083  0.056  0.008  0.398
                   500  0.773  0.149   0.00  0.787  0.846    5.190  0.826  0.813  0.211  1.089  0.120  0.148  0.392
           2.0 1.0 250 -0.027  0.252   0.98  0.251  0.879   -0.107  0.826  0.071  0.077  0.382  0.960  0.916  0.052
                   500  0.047  0.135   0.92  0.142  0.910    0.345  0.826  0.071  0.067  0.300  1.000  1.084  0.050
or_did     0.5 1.0 500  0.234  0.152   1.00  0.258  2.785    1.544  0.826    NaN    NaN    NaN    NaN    NaN    NaN
oracle     0.5 1.0 250 -0.044  0.160   0.96  0.165  1.016   -0.274  0.826    NaN    NaN    NaN    NaN    NaN    NaN
                   500 -0.007  0.123   0.94  0.122  0.958   -0.060  0.82

D3: Changs DGP but with alignment


TypeError: runs.run() got multiple values for keyword argument 'N'

In [33]:
if __name__ == "__main__":
    pd.set_option("display.width", 200)
    kw = dict(R=50, n_jobs=1)
    print(runs.summarize(d4.design_d4(N=500, **kw), by=("dense_g", "dense_l", "est")).round(3))

                             bias     sd  cover   rmse  se_sd  bias_sd  naive     B1  err_g  err_l  sel_g  sel_l  kappa
dense_g dense_l est                                                                                                    
False   False   dml_plugin  0.405  0.215   0.28  0.458  0.635    1.883  0.826  0.438  0.191  0.878  0.992  1.520  0.142
                oracle      0.005  0.104   0.92  0.103  1.083    0.048  0.826    NaN    NaN    NaN    NaN    NaN    NaN
                unadjusted  0.816  0.129   0.00  0.826  0.942    6.335  0.826    NaN    NaN    NaN    NaN    NaN    NaN
        True    dml_plugin  0.274  0.160   0.56  0.317  0.913    1.710  0.370  0.271  0.191  1.039  0.992  0.008  0.142
                oracle      0.005  0.104   0.92  0.103  1.083    0.048  0.370    NaN    NaN    NaN    NaN    NaN    NaN
                unadjusted  0.377  0.128   0.18  0.398  0.978    2.937  0.370    NaN    NaN    NaN    NaN    NaN    NaN
True    False   dml_plugin  0.162  0.163